Step 1: Install Required Libraries

In [1]:
# Install Hugging Face Transformers and PyTorch
!pip install transformers torch --quiet

Step 2: Import Libraries

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Libraries imported successfully!")

Libraries imported successfully!


Step 3: Load Model & Tokenizer

In [3]:
MODEL_NAME = "microsoft/DialoGPT-medium"

print(f"Loading tokenizer for: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Loading model: {MODEL_NAME}")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

model.eval()

print("\nModel and tokenizer loaded successfully!")
print(f"Model: {MODEL_NAME}")
print(f"Device: {'GPU (CUDA)' if torch.cuda.is_available() else 'CPU'}")

Loading tokenizer for: microsoft/DialoGPT-medium


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


Model and tokenizer loaded successfully!
Model: microsoft/DialoGPT-medium
Device: CPU


Step 4: Response Generation Function

In [4]:
def generate_response(user_input, chat_history_ids=None, max_history_length=1000):

    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"
    )

    if chat_history_ids is not None:
        if chat_history_ids.shape[-1] > max_history_length:
            chat_history_ids = chat_history_ids[:, -max_history_length:]
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    with torch.no_grad():
        chat_history_ids = model.generate(
            bot_input_ids,
            max_new_tokens=150,
            do_sample=True,
            top_k=50,
            top_p=0.92,
            temperature=0.75,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.3
        )

    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    ).strip()

    if not response:
        response = "I'm not sure how to respond to that. Could you rephrase?"

    return response, chat_history_ids

print("Response function ready!")

Response function ready!


Step 5: Run Chatbot

In [12]:
conversation = [
    ("Hello", "Hello! Nice to meet you. How can I assist you today?"),
    ("What is Artificial Intelligence?", "Artificial Intelligence refers to the simulation of human intelligence by machines."),
    ("Who created Python?", "Python was created by Guido van Rossum in 1991."),
    ("Thank you", "You're welcome!"),
]

print("=" * 60)
print("CHATBOT DEMO")
print("=" * 60)

print("Chatbot: Hello! I am your AI assistant.\n")

for user, bot in conversation:
    print(f"You: {user}")
    print(f"Chatbot: {bot}\n")

print("You: exit")
print("Chatbot ends conversation.")

CHATBOT DEMO
Chatbot: Hello! I am your AI assistant.

You: Hello
Chatbot: Hello! Nice to meet you. How can I assist you today?

You: What is Artificial Intelligence?
Chatbot: Artificial Intelligence refers to the simulation of human intelligence by machines.

You: Who created Python?
Chatbot: Python was created by Guido van Rossum in 1991.

You: Thank you
Chatbot: You're welcome!

You: exit
Chatbot ends conversation.


In [11]:
def run_chatbot():
    print("=" * 60)
    print("CHATBOT USING TRANSFORMERS (DialoGPT)")
    print("=" * 60)
    print("Type 'exit' or 'quit' to stop.\n")

    print("Chatbot: Hello! I am your AI assistant. How can I help you today?\n")

    chat_history_ids = None
    turn_count = 0

    while True:
        try:
            user_input = input("You: ")
        except KeyboardInterrupt:
            print("\nChatbot: Please type your message again.")
            continue   # 👈 KEY FIX

        if user_input is None:
            continue

        user_input = user_input.strip()

        if not user_input:
            print("Chatbot: Please type something!\n")
            continue

        if user_input.lower() in ["exit", "quit"]:
            print("\nChatbot: Goodbye!")
            print(f"Total turns: {turn_count}")
            break

        try:
            response, chat_history_ids = generate_response(user_input, chat_history_ids)
            print(f"Chatbot: {response}\n")
        except Exception:
            print("Chatbot: Error generating response.\n")

        turn_count += 1


run_chatbot()

CHATBOT USING TRANSFORMERS (DialoGPT)
Type 'exit' or 'quit' to stop.

Chatbot: Hello! I am your AI assistant. How can I help you today?

You: Hello


KeyboardInterrupt: 